# ESM2-CrossFusion v1 — development-only screen

This notebook runs one seed as a go/no-go screen on the protein-grouped development split. It does **not** read the independent test set and does not perform a full-data refit. A GPU runtime is required.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Private-repository access uses a fine-grained read-only GitHub token stored in Colab Secrets as `GITHUB_TOKEN`, with notebook access enabled.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
except ImportError:
    GITHUB_TOKEN = None

git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
askpass_path = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    git_environment['GIT_ASKPASS'] = str(askpass_path)
    git_environment['GITHUB_TOKEN'] = GITHUB_TOKEN

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--recurse-submodules',
            'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
            str(REPO_ROOT),
        ], check=True, env=git_environment)
    else:
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
            check=True, env=git_environment,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
            check=True, env=git_environment,
        )
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Git access failed. Add GITHUB_TOKEN to Colab Secrets for a private repository.'
    ) from error
finally:
    askpass_path.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
print('Project:', REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt

In [ ]:
import json
import platform
import sys
import sklearn
import torch
import transformers

print('Python:', sys.version)
print('Platform:', platform.platform())
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('A GPU runtime is required for this screen.')

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError('Could not open subprocess output stream')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Preflight tests

In [ ]:
run_live([
    sys.executable, '-u', '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
])

## Run seed 42

The run directory is on Google Drive, so its history, best development checkpoint, manifest, and metrics survive a runtime disconnection. The first model download may delay the first epoch briefly.

In [ ]:
from datetime import date

RUN_DIR = (
    Path('/content/drive/MyDrive/MMUbiPred-research/experiments')
    / f'{date.today().isoformat()}-esm2-crossfusion-v1-seed42'
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RUN_DIR / 'validation_metrics.json'

if METRICS_PATH.exists():
    print('Completed metrics already exist; training was skipped:', METRICS_PATH)
else:
    run_live([
        sys.executable, '-u', 'experiment/scripts/train.py',
        '--config', 'experiment/configs/esm2_crossfusion_v1.json',
        '--seed', '42',
        '--output-dir', RUN_DIR,
        '--development-only',
    ])
print('Persistent run directory:', RUN_DIR)

## Inspect the go/no-go result

The stabilized baseline mean fixed-threshold MCC is 0.5639. A single seed cannot establish superiority; it determines only eligibility for a three-seed paired run.

In [ ]:
metrics = json.loads(METRICS_PATH.read_text())
fixed = metrics['fixed_threshold']
selected = metrics['validation_selected_threshold']
display(fixed)
display(selected)
display(metrics['development_selection'])

mcc = float(fixed['mcc'])
if mcc >= 0.56:
    decision = 'COMPETITIVE SCREEN: eligible for protocol review before additional seeds.'
elif mcc >= 0.54:
    decision = 'BORDERLINE SCREEN: advancement requires learning-curve review.'
else:
    decision = 'REJECT: this version does not advance to a three-seed run.'
print(decision)
print('Locked independent test accessed:', False)